In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("cleaned_data.csv")

# 2. Define Regression label
# Using 'TotalCharges' as the continuous numeric target
y_reg = df['TotalCharges']

# 3. Define Classification label
# Using the natural binary column 'Churn' (Mapping 'yes' to 1 and 'no' to 0)
# .str.strip() removes hidden spaces, .str.lower() makes everything lowercase
y_clf = (df['Churn'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# Verify that you now have both 0s and 1s!
print(y_clf.value_counts())


# 4. Define Feature matrix X
# Dropping the target variables and the unique customerID string
X = df.drop(columns=['TotalCharges', 'Churn', 'customerID'])

In [13]:
# 1. Label Encoding for Ordinal Categories
# Defining the natural order for the 'Contract' column
contract_mapping = {
    'month-to-month': 0,
    'one year': 1,
    'two year': 2
}
X['Contract'] = X['Contract'].map(contract_mapping)

# 2. One-Hot Encoding for Nominal Categories
# Defining columns that lack a natural order
nominal_cols = [
    'gender', 
    'PhoneService', 
    'InternetService', 
    'PaperlessBilling', 
    'PaymentMethod'
]

# Apply one-hot encoding and drop the first dummy to avoid multicollinearity
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

# Ensure resulting dummy columns are integers (if pandas outputs booleans)
X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Train-Test Split (80% Training, 20% Testing)
# Splitting X, y_reg, and y_clf simultaneously ensures indices align perfectly
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, 
    y_reg, 
    y_clf, 
    test_size=0.2, 
    random_state=42
)

# 2. Initialize the StandardScaler
scaler = StandardScaler()

# 3. Fit and transform on the TRAINING data ONLY
# This calculates the mean and standard deviation of the training set
X_train_scaled = scaler.fit_transform(X_train)

# 4. Transform the TEST data
# This scales the test set using the mean and standard deviation learned from the training set
X_test_scaled = scaler.transform(X_test)

#  Convert the scaled arrays back to pandas DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 1. Train OLS Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)

# 2. Predict and Evaluate OLS
y_pred_reg_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_reg_test, y_pred_reg_lr)
r2_lr = r2_score(y_reg_test, y_pred_reg_lr)

print("--- OLS Linear Regression ---")
print(f"MSE: {mse_lr:.4f}")
print(f"R²: {r2_lr:.4f}\n")

# 3. Extract and display coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print("Coefficients:")
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

# Identify Top 3 features by absolute magnitude
print("\nTop 3 Features by Absolute Coefficient:")
print(coef_df.head(3)[['Feature', 'Coefficient', 'Abs_Coefficient']].to_string(index=False))

# 4. Train Ridge Regression (alpha=1.0)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)

# 5. Predict and Evaluate Ridge
y_pred_reg_ridge = ridge.predict(X_test_scaled)
mse_ridge = mean_squared_error(y_reg_test, y_pred_reg_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_reg_ridge)

print("\n--- Ridge Regression ---")
print(f"MSE: {mse_ridge:.4f}")
print(f"R²: {r2_ridge:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# 1. Check Class Imbalance
class_proportions = y_clf_train.value_counts(normalize=True)
print(class_proportions)

# 2. Initialize and Train Logistic Regression
# Because the minority class is < 35%, we apply class_weight='balanced'
lr1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr1.fit(X_train_scaled, y_clf_train)

# 3. Predict Class Labels and Probabilities
y_pred_clf = lr1.predict(X_test_scaled)
y_pred_probs = lr1.predict_proba(X_test_scaled)[:, 1]

# 4. Generate Classification Metrics
cm = confusion_matrix(y_clf_test, y_pred_clf)
cr = classification_report(y_clf_test, y_pred_clf)

print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)

# 5. Calculate AUC and Plot ROC Curve
auc_score = roc_auc_score(y_clf_test, y_pred_probs)
print(f"\nAUC Score: {auc_score:.4f}")

fpr, tpr, thresholds = roc_curve(y_clf_test, y_pred_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#1f77b4', linewidth=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.annotate(f'AUC = {auc_score:.3f}', xy=(0.6, 0.2), xytext=(0.6, 0.2), fontsize=14,
             bbox=dict(boxstyle="round,pad=0.3", edgecolor='black', facecolor='white'))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred_probs = lr1.predict_proba(X_test_scaled)[:, 1]

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

print("Threshold | Precision | Recall | F1")
print("---|---|---|---")
for t in thresholds:
    # Convert probabilities to class labels based on the current threshold
    y_pred_t = (y_pred_probs >= t).astype(int)
    
    # Calculate metrics
    p = precision_score(y_clf_test, y_pred_t, zero_division=0)
    r = recall_score(y_clf_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_t, zero_division=0)
    
    # Print row
    print(f"{t:.2f} | {p:.4f} | {r:.4f} | {f1:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# 1. Baseline Model (Default C=1.0)
lr_base = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
lr_base.fit(X_train_scaled, y_clf_train)

y_pred_base = lr_base.predict(X_test_scaled)
y_prob_base = lr_base.predict_proba(X_test_scaled)[:, 1]

# 2. Experimental Model (Strong Regularization C=0.01)
lr_reg = LogisticRegression(C=0.01, class_weight='balanced', max_iter=1000, random_state=42)
lr_reg.fit(X_train_scaled, y_clf_train)

y_pred_reg = lr_reg.predict(X_test_scaled)
y_prob_reg = lr_reg.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate and Compare Metrics
metrics = {
    'C=1.0 (Baseline)': [
        precision_score(y_clf_test, y_pred_base), 
        recall_score(y_clf_test, y_pred_base), 
        roc_auc_score(y_clf_test, y_prob_base)
    ],
    'C=0.01 (Strong Reg)': [
        precision_score(y_clf_test, y_pred_reg), 
        recall_score(y_clf_test, y_pred_reg), 
        roc_auc_score(y_clf_test, y_prob_reg)
    ]
}

print("Metrics Comparison:", metrics)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score


n_bootstraps = 500
auc_diffs = []
y_test_arr = y_clf_test.values
rng = np.random.RandomState(42) 

for _ in range(n_bootstraps):
    # Sample row indices with replacement
    indices = rng.choice(len(y_test_arr), size=len(y_test_arr), replace=True)
    
    y_true_boot = y_test_arr[indices]
    
    # Skip iteration if the bootstrap sample only contains one class (AUC requires both)
    if len(np.unique(y_true_boot)) < 2:
        continue
        
    # Index predicted probabilities using the same sampled indices
    y_prob_base_boot = y_prob_base[indices]
    y_prob_reg_boot = y_prob_reg[indices]
    
    # Compute AUC for both models on the bootstrap sample
    auc_base_boot = roc_auc_score(y_true_boot, y_prob_base_boot)
    auc_reg_boot = roc_auc_score(y_true_boot, y_prob_reg_boot)
    
    # Compute and store the difference (C=1.0 - C=0.01)
    auc_diffs.append(auc_base_boot - auc_reg_boot)

# Calculate the mean and the 95% Confidence Interval (2.5th and 97.5th percentiles)
mean_diff = np.mean(auc_diffs)
ci_lower = np.percentile(auc_diffs, 2.5)
ci_upper = np.percentile(auc_diffs, 97.5)

print(f"Mean AUC Difference: {mean_diff:.5f}")
print(f"95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize an unconstrained Decision Tree (max_depth=None by default)
dt_baseline = DecisionTreeClassifier(random_state=42)

# 2. Fit the model on the scaled training data
dt_baseline.fit(X_train_scaled, y_clf_train)

# 3. Calculate Accuracy for both Training and Testing Sets
train_acc = accuracy_score(y_clf_train, dt_baseline.predict(X_train_scaled))
test_acc = accuracy_score(y_clf_test, dt_baseline.predict(X_test_scaled))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


# Initialize the controlled Decision Tree with the requested hyperparameters
dt_controlled = DecisionTreeClassifier(
    max_depth=5, 
    min_samples_split=20, 
    random_state=42
)

# Train the model on the scaled training data
dt_controlled.fit(X_train_scaled, y_clf_train)

# Generate predictions
y_train_pred_ctrl = dt_controlled.predict(X_train_scaled)
y_test_pred_ctrl = dt_controlled.predict(X_test_scaled)

# Calculate and report the accuracies
train_accuracy_ctrl = accuracy_score(y_clf_train, y_train_pred_ctrl)
test_accuracy_ctrl = accuracy_score(y_clf_test, y_test_pred_ctrl)

# 1. Generate predicted probabilities for the test set
# We slice [:, 1] to get the probabilities of the positive class (Churn = 1)
y_test_prob_dt = dt_controlled.predict_proba(X_test_scaled)[:, 1]

# 2. Calculate the ROC-AUC score
roc_auc_dt = roc_auc_score(y_clf_test, y_test_prob_dt)

# 3. Display the result
print(f"Controlled Decision Tree Test ROC-AUC: {roc_auc_dt:.4f}")

print(f"Controlled Decision Tree - Training Accuracy: {train_accuracy_ctrl:.4f}")
print(f"Controlled Decision Tree - Testing Accuracy: {test_accuracy_ctrl:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Initialize the Gini Decision Tree
dt_gini = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5, 
    random_state=42
)

# 2. Initialize the Entropy Decision Tree
dt_entropy = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5, 
    random_state=42
)

# 3. Train both models
dt_gini.fit(X_train_scaled, y_clf_train)
dt_entropy.fit(X_train_scaled, y_clf_train)

# 4. Generate predictions
y_test_pred_gini = dt_gini.predict(X_test_scaled)
y_test_pred_entropy = dt_entropy.predict(X_test_scaled)

# 5. Calculate and report accuracies
test_accuracy_gini = accuracy_score(y_clf_test, y_test_pred_gini)
test_accuracy_entropy = accuracy_score(y_clf_test, y_test_pred_entropy)

print(f"Gini Decision Tree Test Accuracy: {test_accuracy_gini:.4f}")
print(f"Entropy Decision Tree Test Accuracy: {test_accuracy_entropy:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Initialize and Train the Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

# 2. Generate Predictions
y_train_pred_rf = rf.predict(X_train_scaled)
y_test_pred_rf = rf.predict(X_test_scaled)
y_test_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate and Report Metrics
train_acc_rf = accuracy_score(y_clf_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_clf_test, y_test_pred_rf)
roc_auc_rf = roc_auc_score(y_clf_test, y_test_prob_rf)

print("--- Random Forest Performance ---")
print(f"Training Accuracy: {train_acc_rf:.4f}")
print(f"Test Accuracy: {test_acc_rf:.4f}")
print(f"ROC-AUC: {roc_auc_rf:.4f}\n")

# 4. Extract and Display Top 5 Features by Importance
feature_importances = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Importance': rf.feature_importances_
})

# Sort descending and get the top 5
top_5_features = feature_importances.sort_values(by='Importance', ascending=False).head(5)

print("--- Top 5 Features by Importance ---")
print(top_5_features.to_string(index=False))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score
import numpy as np

# 1. Initialize and Train the Gradient Boosting Classifier
gb = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)
gb.fit(X_train_scaled, y_clf_train)

# 2. Generate Predictions
y_train_pred_gb = gb.predict(X_train_scaled)
y_test_pred_gb = gb.predict(X_test_scaled)
y_test_prob_gb = gb.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate and Report Standard Metrics
train_acc_gb = accuracy_score(y_clf_train, y_train_pred_gb)
test_acc_gb = accuracy_score(y_clf_test, y_test_pred_gb)
roc_auc_gb = roc_auc_score(y_clf_test, y_test_prob_gb)

print("--- Gradient Boosting Performance ---")
print(f"Training Accuracy: {train_acc_gb:.4f}")
print(f"Test Accuracy: {test_acc_gb:.4f}")
print(f"ROC-AUC: {roc_auc_gb:.4f}\n")



In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Identify the 5 lowest-importance features from the original Random Forest
feature_importances = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Importance': rf.feature_importances_
})

# Sort ascending to get the least important features at the top
lowest_5_features = feature_importances.sort_values(by='Importance', ascending=True).head(5)
features_to_drop = lowest_5_features['Feature'].tolist()

print("--- 5 Features with Lowest Importance ---")
print(lowest_5_features.to_string(index=False))

#  Create reduced datasets by dropping the bottom 5 features
X_train_reduced = X_train_scaled.drop(columns=features_to_drop)
X_test_reduced = X_test_scaled.drop(columns=features_to_drop)

# Train the reduced Random Forest with identical hyperparameters
rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)

# Generate predictions and evaluate the reduced model
y_test_prob_rf_reduced = rf_reduced.predict_proba(X_test_reduced)[:, 1]
roc_auc_rf_reduced = roc_auc_score(y_clf_test, y_test_prob_rf_reduced)


print("\n--- Feature Ablation Results ---")
print(f"Full Model Test ROC-AUC: {roc_auc_rf:.4f}")
print(f"Reduced Model Test ROC-AUC: {roc_auc_rf_reduced:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# 1. Define the cross-validation strategy
# StratifiedKFold ensures the ratio of churners to non-churners remains constant in every fold
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Group the models into a dictionary for easy iteration
# Ensure these variable names match what you initialized in previous cells
models = {
    'Logistic Regression': lr1, 
    'Decision Tree (max_depth=5)': dt_controlled,
    'Random Forest': rf,
    'Gradient Boosting': gb
}

# 3. Evaluate each model and print the results
print("--- 5-Fold Cross-Validation Results (ROC-AUC) ---")
for name, model in models.items():
    # Compute cross-validated AUC scores on the training data
    cv_scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv_strategy, scoring='roc_auc')
    
    # Calculate mean and standard deviation
    mean_auc = np.mean(cv_scores)
    std_auc = np.std(cv_scores)
    
    # Report the metrics
    print(f"{name:<28}: Mean AUC = {mean_auc:.4f} | Std Dev = {std_auc:.4f}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# 1. Define the Pipeline
# The pipeline automatically handles imputation and scaling before passing data to the model
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# 2. Define the Parameter Grid
# The double underscore '__' links the parameter to the specific step in the pipeline
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}

# 3. Set up the Cross-Validation Strategy and GridSearchCV
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=cv_strategy, 
    scoring='roc_auc', 
    n_jobs=-1  # -1 utilizes all available CPU cores to speed up the process
)

# 4. Execute the Grid Search on the UNSCALED training data
print("Running GridSearchCV... This may take a moment.")
grid_search.fit(X_train, y_clf_train)

# Get the index of the best-performing model in the grid search
best_index = grid_search.best_index_

# Extract the standard deviation of the CV scores for that specific model
std_auc_tuned_rf = grid_search.cv_results_['std_test_score'][best_index]

# 5. Extract and print the best results
print("\n--- GridSearchCV Results ---")
print(f"Best ROC-AUC Score: {grid_search.best_score_:.4f}")
print(f"Tuned Random Forest 5-Fold CV Std AUC: {std_auc_tuned_rf:.4f}")
print("Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

# Extract the best model from your completed grid_search
best_rf_model = grid_search.best_estimator_

# Define the fractions of the dataset to train on
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

print(f"{'Training Fraction':<20} | {'Training AUC':<15} | {'Test AUC':<15}")
print("-" * 55)

for f in fractions:
    # 1. Determine the subset size
    subset_size = int(f * len(X_train))
    
    # 2. Slice the training data
    X_subset = X_train.iloc[:subset_size]
    y_subset = y_clf_train.iloc[:subset_size]
    
    # 3. Fit the pipeline on the subset
    best_rf_model.fit(X_subset, y_subset)
    
    # 4. Predict probabilities for Training subset and Full Test set
    y_train_prob = best_rf_model.predict_proba(X_subset)[:, 1]
    y_test_prob = best_rf_model.predict_proba(X_test)[:, 1] 
    
    # 5. Calculate AUC scores
    train_auc = roc_auc_score(y_subset, y_train_prob)
    test_auc = roc_auc_score(y_clf_test, y_test_prob)
    
    # 6. Print the row
    print(f"{f:<20} | {train_auc:<15.4f} | {test_auc:<15.4f}")

In [ ]:
import joblib
import pandas as pd

# 1. Save the best pipeline to disk
# best_rf_model contains the SimpleImputer, StandardScaler, and RandomForest
joblib.dump(best_rf_model, 'best_model.pkl')
print("Model successfully saved to 'best_model.pkl'")

# 2. Hand-craft two test rows
# Grabbing 2 rows from X_test ensures we have the exact right columns and data types
hand_crafted_data = X_test.iloc[0:2].copy()

# Manually engineering Customer A: High churn risk profile
hand_crafted_data.iloc[0, hand_crafted_data.columns.get_loc('tenure')] = 1
hand_crafted_data.iloc[0, hand_crafted_data.columns.get_loc('Contract')] = 0 # month-to-month
hand_crafted_data.iloc[0, hand_crafted_data.columns.get_loc('MonthlyCharges')] = 105.50

# Manually engineering Customer B: Low churn risk profile
hand_crafted_data.iloc[1, hand_crafted_data.columns.get_loc('tenure')] = 72
hand_crafted_data.iloc[1, hand_crafted_data.columns.get_loc('Contract')] = 2 # two-year
hand_crafted_data.iloc[1, hand_crafted_data.columns.get_loc('MonthlyCharges')] = 20.00

# 3. Load the saved model from disk
loaded_model = joblib.load('best_model.pkl')

# 4. Generate predictions on the new payload
predictions = loaded_model.predict(hand_crafted_data)
probabilities = loaded_model.predict_proba(hand_crafted_data)[:, 1]

print("\n--- Inference Results from Reloaded Model ---")
print(f"Customer A (High Risk Profile) - Predicted Churn: {predictions[0]} (Probability: {probabilities[0]:.4f})")
print(f"Customer B (Low Risk Profile)  - Predicted Churn: {predictions[1]} (Probability: {probabilities[1]:.4f})")